In [5]:
from google.colab import files
uploaded = files.upload()

Saving dataset.zip to dataset.zip


In [6]:
import os
print(os.listdir('/content'))

['.config', 'dataset.zip', 'sample_data']


In [7]:
import zipfile

zip_path = '/content/dataset.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print("✅ Extracted successfully")

✅ Extracted successfully


In [8]:
import os
print(os.listdir('/content/dataset'))

['dataset']


In [10]:
dataset_path = '/content/dataset/dataset'

In [11]:
from PIL import Image
import os

def clean_dataset(path):
    removed = 0
    for root, dirs, files in os.walk(path):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path).convert('RGB')
                img.verify()
            except:
                os.remove(file_path)
                removed += 1
    print("✅ Removed bad images:", removed)

clean_dataset(dataset_path)

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


✅ Removed bad images: 0


In [12]:
import tensorflow as tf

train_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

class_names = train_data.class_names
print("Classes:", class_names)

Found 698 files belonging to 8 classes.
Using 559 files for training.
Found 698 files belonging to 8 classes.
Using 139 files for validation.
Classes: ['IC', 'arduino', 'capacitor', 'jumper wire', 'led', 'push_button', 'resistor', 'ultrasonic_sensor']


In [13]:
AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_data = val_data.cache().prefetch(buffer_size=AUTOTUNE)

In [14]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import tensorflow as tf

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')
])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [15]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [16]:
import tensorflow as tf

img_size = (224, 224)
batch_size = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    color_mode='rgb'   # 🔥 FORCE RGB (fixes your error)
)

val_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    color_mode='rgb'   # 🔥 IMPORTANT
)

Found 698 files belonging to 8 classes.
Using 559 files for training.
Found 698 files belonging to 8 classes.
Using 139 files for validation.


In [17]:
def fix_images(image, label):
    image = tf.image.resize(image, (224, 224))
    image = tf.cast(image, tf.float32)
    return image, label

train_data = train_data.map(fix_images)
val_data = val_data.map(fix_images)

In [18]:
AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.prefetch(buffer_size=AUTOTUNE)
val_data = val_data.prefetch(buffer_size=AUTOTUNE)

In [19]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [20]:
import os
from PIL import Image

clean_path = '/content/clean_dataset'

if not os.path.exists(clean_path):
    os.makedirs(clean_path)

In [21]:
import shutil

for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)

    if not os.path.isdir(class_path):
        continue

    new_class_path = os.path.join(clean_path, class_name)
    os.makedirs(new_class_path, exist_ok=True)

    for file in os.listdir(class_path):
        old_file = os.path.join(class_path, file)
        new_file = os.path.join(new_class_path, file)

        try:
            img = Image.open(old_file).convert('RGB')
            img.save(new_file, 'JPEG')   # ✅ force clean format
        except:
            print("Skipped bad file:", old_file)

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [22]:
dataset_path = '/content/clean_dataset'

import tensorflow as tf

train_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

Found 698 files belonging to 8 classes.
Using 559 files for training.
Found 698 files belonging to 8 classes.
Using 139 files for validation.


In [23]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [24]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - accuracy: 0.5403 - loss: 1.3255 - val_accuracy: 0.7338 - val_loss: 0.7209
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.7961 - loss: 0.6295 - val_accuracy: 0.7914 - val_loss: 0.6315
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.8819 - loss: 0.4101 - val_accuracy: 0.8058 - val_loss: 0.5065
Epoch 4/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.9141 - loss: 0.3152 - val_accuracy: 0.8561 - val_loss: 0.4798
Epoch 5/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 103ms/step - accuracy: 0.9374 - loss: 0.2345 - val_accuracy: 0.8561 - val_loss: 0.4381
Epoch 6/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.9660 - loss: 0.1818 - val_accuracy: 0.8849 - val_loss: 0.3800
Epoch 7/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.9624 - loss: 0.1593 - val_accuracy: 0.8849 - val_loss: 0.3881
Epoch 8/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.9678 - loss: 0.1377 - val_accuracy: 0.8633 - v

In [25]:
model.save('/content/component_model.keras')

In [30]:
def predict_image(img_path):
    from tensorflow.keras.preprocessing import image
    import numpy as np
    import matplotlib.pyplot as plt

    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    prediction = model.predict(img_array)

    class_names = [
        'resistor','capacitor','diode','inductor',
        'ic','transistor','led','switch'
    ]

    predicted_index = np.argmax(prediction)
    confidence = np.max(prediction)

    predicted_class = class_names[predicted_index]

    plt.imshow(img)
    plt.axis('off')
    plt.title(f"{predicted_class.upper()} ({confidence*100:.2f}%)")
    plt.show()

    print(f"Component: {predicted_class}")
    print(f"Confidence: {confidence*100:.2f}%")

In [31]:
import os
print(os.listdir('/content'))

['.config', 'dataset.zip', 'component_model.keras', 'dataset', 'clean_dataset', 'sample_data']


In [32]:
from google.colab import files
uploaded = files.upload()

Saving dataset.zip to dataset (1).zip


In [33]:
import zipfile

with zipfile.ZipFile('/content/dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print("✅ Extracted")

✅ Extracted


In [34]:
!ls /content/dataset

dataset


In [35]:
!ls /content/dataset/dataset

 arduino     IC		    led		  resistor
 capacitor  'jumper wire'   push_button   ultrasonic_sensor


In [37]:
!ls /content/dataset

dataset


In [38]:
!ls /content/dataset/dataset

 arduino     IC		    led		  resistor
 capacitor  'jumper wire'   push_button   ultrasonic_sensor


In [39]:
DATASET_PATH = '/content/dataset/dataset'

In [40]:
DATASET_PATH = '/content/dataset'

In [41]:
DATASET_PATH = '/content/dataset/dataset'   # or '/content/dataset'

In [42]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='sparse',
    subset='training'
)

Found 562 images belonging to 8 classes.


In [43]:
print(train_data.class_indices)

{'IC': 0, 'arduino': 1, 'capacitor': 2, 'jumper wire': 3, 'led': 4, 'push_button': 5, 'resistor': 6, 'ultrasonic_sensor': 7}


In [44]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 22s 839ms/step - accuracy: 0.7082 - loss: 1.0431 - val_accuracy: 0.8273 - val_loss: 0.4825
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 152ms/step - accuracy: 0.9502 - loss: 0.1381 - val_accuracy: 0.8489 - val_loss: 0.4930
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 146ms/step - accuracy: 0.9982 - loss: 0.0451 - val_accuracy: 0.8345 - val_loss: 0.4468
Epoch 4/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 185ms/step - accuracy: 0.9982 - loss: 0.0211 - val_accuracy: 0.8417 - val_loss: 0.4565
Epoch 5/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 153ms/step - accuracy: 1.0000 - loss: 0.0117 - val_accuracy: 0.8417 - val_loss: 0.4614
Epoch 6/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 150ms/step - accuracy: 1.0000 - loss: 0.0081 - val_accuracy: 0.8489 - val_loss: 0.4633
Epoch 7/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 146ms/step - accuracy: 1.0000 - loss: 0.0066 - val_accuracy: 0.8417 - val_loss: 0.4683
Epoch 8/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 190ms/step - accuracy: 1.0000 - loss: 0.0056 - val_accuracy: 0

In [45]:
import tensorflow as tf

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(8, activation='softmax')
])

In [47]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [48]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 25s 888ms/step - accuracy: 0.6370 - loss: 1.0512 - val_accuracy: 0.1367 - val_loss: 2.6779
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 184ms/step - accuracy: 0.9270 - loss: 0.2490 - val_accuracy: 0.1655 - val_loss: 3.1542
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 160ms/step - accuracy: 0.9520 - loss: 0.1416 - val_accuracy: 0.2014 - val_loss: 3.1439
Epoch 4/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 150ms/step - accuracy: 0.9840 - loss: 0.0674 - val_accuracy: 0.1727 - val_loss: 3.5791
Epoch 5/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 151ms/step - accuracy: 0.9982 - loss: 0.0361 - val_accuracy: 0.1799 - val_loss: 3.6935
Epoch 6/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 146ms/step - accuracy: 0.9911 - loss: 0.0365 - val_accuracy: 0.1799 - val_loss: 3.9070
Epoch 7/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 190ms/step - accuracy: 0.9964 - loss: 0.0265 - val_accuracy: 0.1367 - val_loss: 3.9335
Epoch 8/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 149ms/step - accuracy: 0.9964 - loss: 0.0226 - val_accuracy: 0

In [49]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='sparse',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='sparse',
    subset='validation'
)

Found 562 images belonging to 8 classes.
Found 136 images belonging to 8 classes.


In [50]:
print(train_data)
print(val_data)

In [51]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 1.0000 - loss: 0.0128 - val_accuracy: 0.9485 - val_loss: 0.1665
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 295ms/step - accuracy: 1.0000 - loss: 0.0075 - val_accuracy: 0.9485 - val_loss: 0.1576
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 170ms/step - accuracy: 1.0000 - loss: 0.0095 - val_accuracy: 0.9485 - val_loss: 0.1870
Epoch 4/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 227ms/step - accuracy: 0.9982 - loss: 0.0080 - val_accuracy: 0.9485 - val_loss: 0.1699
Epoch 5/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 157ms/step - accuracy: 0.9982 - loss: 0.0111 - val_accuracy: 0.9485 - val_loss: 0.1847
Epoch 6/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 148ms/step - accuracy: 1.0000 - loss: 0.0082 - val_accuracy: 0.9485 - val_loss: 0.1735
Epoch 7/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 147ms/step - accuracy: 1.0000 - loss: 0.0076 - val_accuracy: 0.9412 - val_loss: 0.1952
Epoch 8/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 173ms/step - accuracy: 1.0000 - loss: 0.0048 - val_accuracy: 0.94

In [52]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.5, 1.5]
)

In [54]:
base_model.trainable = True

# freeze first layers, train last layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [55]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [56]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.9466 - loss: 0.1334 - val_accuracy: 0.9338 - val_loss: 0.2158
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 175ms/step - accuracy: 0.9858 - loss: 0.0404 - val_accuracy: 0.9338 - val_loss: 0.1842
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 159ms/step - accuracy: 1.0000 - loss: 0.0149 - val_accuracy: 0.9338 - val_loss: 0.1694
Epoch 4/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 194ms/step - accuracy: 0.9964 - loss: 0.0129 - val_accuracy: 0.9338 - val_loss: 0.1679
Epoch 5/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 168ms/step - accuracy: 1.0000 - loss: 0.0076 - val_accuracy: 0.9485 - val_loss: 0.1771
Epoch 6/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 152ms/step - accuracy: 0.9982 - loss: 0.0082 - val_accuracy: 0.9559 - val_loss: 0.1711
Epoch 7/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 158ms/step - accuracy: 1.0000 - loss: 0.0034 - val_accuracy: 0.9559 - val_loss: 0.1650
Epoch 8/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 198ms/step - accuracy: 0.9982 - loss: 0.0056 - val_accuracy: 0.95

In [59]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stop]
)

Epoch 1/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 181ms/step - accuracy: 1.0000 - loss: 9.7646e-04 - val_accuracy: 0.9706 - val_loss: 0.1175
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 162ms/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 0.9706 - val_loss: 0.1137
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 158ms/step - accuracy: 1.0000 - loss: 0.0017 - val_accuracy: 0.9706 - val_loss: 0.1120
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 192ms/step - accuracy: 1.0000 - loss: 5.4075e-04 - val_accuracy: 0.9779 - val_loss: 0.1147
Epoch 5/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 160ms/step - accuracy: 1.0000 - loss: 0.0013 - val_accuracy: 0.9706 - val_loss: 0.1232
Epoch 6/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 162ms/step - accuracy: 1.0000 - loss: 0.0021 - val_accuracy: 0.9706 - val_loss: 0.1233


In [61]:
img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)

# 🔥 better normalization for MobileNet
img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)

img_array = np.expand_dims(img_array, axis=0)

FileNotFoundError: [Errno 2] No such file or directory: '/content/download (3).jpeg'

In [62]:
if confidence < 0.7:
    print("⚠️ Not confident prediction")

NameError: name 'confidence' is not defined

In [78]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = '📸 Capture';
      div.appendChild(button);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      await new Promise((resolve) => button.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getTracks().forEach(track => track.stop());
      div.remove();

      return canvas.toDataURL('image/jpeg', quality);
    }
  ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])

  with open(filename, 'wb') as f:
    f.write(binary)

  return filename

In [ ]:
img_path = take_photo()
print("Saved:", img_path)

<IPython.core.display.Javascript object>

In [80]:
import tensorflow as tf

model = tf.keras.models.load_model('/content/component_model.h5')

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/content/component_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [81]:
import numpy as np
from tensorflow.keras.preprocessing import image

img = image.load_img(img_path)

# 🔥 Crop center (removes background noise)
width, height = img.size
left = width * 0.2
top = height * 0.2
right = width * 0.8
bottom = height * 0.8

img = img.crop((left, top, right, bottom))

# Resize
img = img.resize((224,224))

# Convert
img_array = image.img_to_array(img)

# 🔥 Use MobileNet preprocessing (IMPORTANT)
img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)

img_array = np.expand_dims(img_array, axis=0)

In [82]:
class_names = ['IC','arduino','capacitor','jumper wire','led','push_button','resistor','ultrasonic_sensor']

In [83]:
prediction = model.predict(img_array)

predicted_index = np.argmax(prediction)
predicted_class = class_names[predicted_index]
confidence = prediction[0][predicted_index]

print("Prediction:", predicted_class)
print("Confidence:", confidence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Prediction: ultrasonic_sensor
Confidence: 0.99999654
